<a href="https://colab.research.google.com/github/lestojas/segmentation/blob/matanglawin-dataset/colab/test_direction_classification_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test crack direction classification models — Part C evaluation

Evaluates the three already-trained YOLO classification checkpoints
(`yolo11n-cls`, `yolo11m-cls`, `yolov8n-cls`) on the **89-image held-out test
set** used in Part C of the Matanglawin paper, and reproduces:

- a **confusion matrix per model** (Horizontal / Vertical / Diagonal / Mixed),
  the same kind of table as Part A's Table 5, and
- the **Table 9 metrics** (Top-1 accuracy, macro precision, macro recall,
  macro F1-score), plus a full per-class precision/recall/F1 breakdown for
  the discussion section.

At the end there is a single block of plain-text output labeled
**"COPY EVERYTHING BELOW THIS LINE"** — copy that and send it back so the
Part C write-up can be updated with the real numbers.

**Runtime:** CPU is fine (89 images × 3 small classifiers), but
`Runtime > Change runtime type > GPU` will make it faster.

## 0. Setup

In [ ]:
!pip install -q ultralytics scikit-learn seaborn pandas

import re
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from ultralytics import YOLO

## 1. Get the 89-image held-out test set

Clones the dataset repo (shallow, single branch) to pull `test/` — the 89
crack-containing test images plus `_direction_labels.csv`, the ground-truth
direction label for each one. This is the exact same held-out split Part A
and Part C were evaluated on.

In [ ]:
REPO_URL = 'https://github.com/lestojas/segmentation.git'
BRANCH   = 'matanglawin-dataset'  # switch to 'main' once this branch is merged

DATA_DIR   = Path('/content/segmentation')
TEST_DIR   = DATA_DIR / 'test'
LABELS_CSV = TEST_DIR / '_direction_labels.csv'

if not LABELS_CSV.exists():
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(DATA_DIR)],
        check=True,
    )

assert LABELS_CSV.exists(), (
    f'{LABELS_CSV} not found. If the repo is private and the clone above failed, '
    "zip your local test/ folder (images + _direction_labels.csv) and upload it "
    "with the cell below instead."
)
print(f'Dataset ready at {DATA_DIR}')

**Only run this next cell if the clone above failed** (e.g. private repo,
no internet). It lets you upload a zip of your local `test/` folder
(containing the images and `_direction_labels.csv`) instead.

In [ ]:
from google.colab import files

_uploaded = files.upload()  # select a .zip of your test/ folder
for _fname in _uploaded:
    if _fname.lower().endswith('.zip'):
        shutil.unpack_archive(_fname, str(TEST_DIR))
        print(f'Unpacked {_fname} into {TEST_DIR}')

assert LABELS_CSV.exists(), f'{LABELS_CSV} still not found after upload.'

## 2. Upload the trained classification checkpoints

Two ways to get `best (yolo11n-cls).pt`, `best (yolo11m-cls).pt`, and
`best (yolov8n-cls).pt` into this Colab session — **run only one of the two
cells below**, not both.

- **Option A — browser upload** (next cell): a file picker pops up: select
  all three `.pt` files together from wherever you saved them on your
  computer/phone.
- **Option B — Google Drive** (cell after that): if the three files already
  live in your Drive (as in your screenshot), mount Drive instead — no
  re-upload needed, and it's faster for larger files.

Filenames are matched loosely (spaces/parentheses ignored), so the exact
names from your Drive are fine as-is.

In [ ]:
MODEL_KEYS = ['yolo11n-cls', 'yolo11m-cls', 'yolov8n-cls']

def _normalize(s):
    return re.sub(r'[^a-z0-9]', '', s.lower())

WEIGHTS_DIR = Path('/content/weights')
WEIGHTS_DIR.mkdir(exist_ok=True)

**Option A — browser upload.** A file picker will open: select all three `.pt` files at once.

In [ ]:
from google.colab import files

print('Upload the three .pt checkpoints (yolo11n-cls, yolo11m-cls, yolov8n-cls):')
uploaded = files.upload()

model_paths = {}
for fname, content in uploaded.items():
    dest = WEIGHTS_DIR / fname
    dest.write_bytes(content)
    norm = _normalize(fname)
    match = next((k for k in MODEL_KEYS if _normalize(k) in norm), None)
    if match:
        model_paths[match] = dest

missing = [k for k in MODEL_KEYS if k not in model_paths]
if missing:
    raise RuntimeError(
        f'Could not match an uploaded file to: {missing}. Uploaded files were: '
        f'{list(uploaded.keys())}. Filenames must contain e.g. "yolo11n-cls".'
    )

for key in MODEL_KEYS:
    print(f'{key}: {model_paths[key].name}')

**Option B — Google Drive.** Mount your Drive, then point `DRIVE_WEIGHTS_DIR` at the folder that contains the three `.pt` files (edit the path to match your Drive).

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_WEIGHTS_DIR = Path('/content/drive/MyDrive')  # <-- edit this to the folder holding the .pt files

model_paths = {}
for f in DRIVE_WEIGHTS_DIR.glob('*.pt'):
    norm = _normalize(f.name)
    match = next((k for k in MODEL_KEYS if _normalize(k) in norm), None)
    if match:
        model_paths[match] = f

missing = [k for k in MODEL_KEYS if k not in model_paths]
if missing:
    raise RuntimeError(
        f'Could not find weights for: {missing} under {DRIVE_WEIGHTS_DIR}. '
        f'Files found there: {[p.name for p in DRIVE_WEIGHTS_DIR.glob("*.pt")]}. '
        'Double-check DRIVE_WEIGHTS_DIR points at the right folder.'
    )

for key in MODEL_KEYS:
    print(f'{key}: {model_paths[key]}')

## 3. Load ground-truth direction labels

`direction` is one of Horizontal / Vertical / Diagonal / Mixed, computed
from the segmentation ground truth (see `DIRECTION_LABELS.md` in the
repo) — this is the label each classifier is being tested against.

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
print(f'{len(labels_df)} test images with ground-truth direction labels')
print(labels_df['direction'].value_counts())

image_paths = [TEST_DIR / fn for fn in labels_df['file_name']]
missing_imgs = [p for p in image_paths if not p.exists()]
assert not missing_imgs, f'{len(missing_imgs)} listed image(s) missing from {TEST_DIR}: {missing_imgs[:5]}'

y_true = labels_df['direction'].tolist()
LABELS_ORDER = sorted(labels_df['direction'].unique())
print('\nClass order used for confusion matrices:', LABELS_ORDER)

## 4. Run inference with each model on all 89 test images

`imgsz=224` matches the size used for training/evaluating the `-cls`
models in `colab/train_crack_direction_model.ipynb` and in
`scripts/inference.py`.

In [ ]:
IMG_SIZE = 224

results_by_model = {}
for key in MODEL_KEYS:
    print(f'Running {key} on {len(image_paths)} test images...')
    model = YOLO(str(model_paths[key]))
    preds = model.predict(source=[str(p) for p in image_paths], imgsz=IMG_SIZE, verbose=False)
    class_names = model.names
    y_pred = [class_names[int(p.probs.top1)] for p in preds]
    y_conf = [float(p.probs.top1conf) for p in preds]
    results_by_model[key] = {'y_pred': y_pred, 'y_conf': y_conf}
    print(f'  done — predicted class distribution: {pd.Series(y_pred).value_counts().to_dict()}')

## 5. Confusion matrix per model

Same evaluation as Part A (Table 5), extended from a 2×2 crack/no-crack
matrix to a 4×4 Horizontal/Vertical/Diagonal/Mixed matrix, since direction
classification has four classes instead of two.

In [ ]:
RESULTS_DIR = Path('/content/results')
RESULTS_DIR.mkdir(exist_ok=True)

cm_tables = {}
report_tables = {}

fig, axes = plt.subplots(1, len(MODEL_KEYS), figsize=(5.5 * len(MODEL_KEYS), 4.5))
axes = np.atleast_1d(axes)

for ax, key in zip(axes, MODEL_KEYS):
    y_pred = results_by_model[key]['y_pred']

    cm = confusion_matrix(y_true, y_pred, labels=LABELS_ORDER)
    cm_df = pd.DataFrame(
        cm,
        index=[f'Actual {l}' for l in LABELS_ORDER],
        columns=[f'Pred {l}' for l in LABELS_ORDER],
    )
    cm_tables[key] = cm_df
    cm_df.to_csv(RESULTS_DIR / f'confusion_matrix_{key}.csv')

    report = classification_report(
        y_true, y_pred, labels=LABELS_ORDER, output_dict=True, digits=4, zero_division=0
    )
    report_df = pd.DataFrame(report).T
    report_tables[key] = report_df
    report_df.to_csv(RESULTS_DIR / f'classification_report_{key}.csv')

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=LABELS_ORDER, yticklabels=LABELS_ORDER)
    ax.set_title(key)
    ax.set_xlabel('Predicted direction')
    ax.set_ylabel('True direction')

plt.suptitle('Crack Direction Classification — Confusion Matrix per Model (n = 89)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'direction_confusion_matrices.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Metrics table (Table 9 format)

Top-1 accuracy, macro precision, macro recall, macro F1-score — the same
four rows and the same `yolo11n-cls` / `yolo11m-cls` / `yolov8n-cls`
column order as the existing Table 9 in the paper, so these numbers can
drop straight in.

In [ ]:
summary = {'Top-1 accuracy': {}, 'Macro precision': {}, 'Macro recall': {}, 'Macro F1-score': {}}

for key in MODEL_KEYS:
    report = report_tables[key]
    accuracy = (np.array(y_true) == np.array(results_by_model[key]['y_pred'])).mean()
    macro = report.loc['macro avg']
    summary['Top-1 accuracy'][key] = accuracy
    summary['Macro precision'][key] = macro['precision']
    summary['Macro recall'][key] = macro['recall']
    summary['Macro F1-score'][key] = macro['f1-score']

table9 = pd.DataFrame(summary).T[MODEL_KEYS]
table9.to_csv(RESULTS_DIR / 'table9_direction_classification_metrics.csv')

print('Table 9. Direction Classification Performance Comparison of YOLO-Based Models')
display((table9 * 100).round(2).astype(str) + '%')

## 7. Copy-paste-ready results block

Everything needed to update Part C: the confusion matrix, the full
per-class precision/recall/F1 report, and the Table 9 summary for each
model. Copy the block below (from the `====` line onward) and send it
back.

In [ ]:
print('=' * 70)
print('COPY EVERYTHING BELOW THIS LINE AND SEND IT BACK')
print('=' * 70)
print(f'\nTest set: n = {len(y_true)} crack-containing images')
print(f'Classes: {LABELS_ORDER}')
print(f'Ground-truth class counts: {pd.Series(y_true).value_counts().to_dict()}\n')

for key in MODEL_KEYS:
    print(f'--- {key} ---')
    print('Confusion matrix (rows = actual, columns = predicted):')
    print(cm_tables[key].to_string())
    print()
    print('Per-class precision / recall / F1-score / support:')
    print(report_tables[key].round(4).to_string())
    print()

print('Table 9 summary:')
print((table9 * 100).round(2).to_string())
print('=' * 70)

## 8. Download all results (CSVs + confusion matrix figure)

In [ ]:
zip_path = shutil.make_archive('/content/direction_classification_results', 'zip', str(RESULTS_DIR))
print(f'Zipped results to {zip_path}')

from google.colab import files
files.download(zip_path)